# FLIR UAP — Análise de Movimento da Câmera

Este notebook estima a **posição, velocidade e aceleração** do sensor de câmera FLIR
a partir do movimento do **fundo** de vídeos de UAP capturados por sistemas de armas
(drones ou caças).

## Contexto

- O sistema de armas mantém o alvo (UAP) próximo ao centro da mira.  
- A câmera é **térmica** (FLIR): o alvo aparece com alto contraste — geralmente
  claro, mas pode ser escuro em alguns vídeos.  
- Elementos fixos da HUD (mira, retículo, crosshair e sobreposições de texto) são
  **mascarados automaticamente** antes da estimativa de movimento.  
- O movimento é estimado por **correlação de fase** no background restante, com
  fluxo óptico denso como fallback.

## Saídas

| Gráfico | Descrição |
|---|---|
| `plot_camera_kinematics()` | Posição, velocidade e aceleração da câmera em X e Y |
| `plot_camera_trajectory()` | Trajetória 2-D da câmera colorida por tempo |
| `plot_motion_quality()` | Métricas de qualidade da estimativa de movimento |

## Convenção de sinal

- `cam_pos_x_px > 0` → câmera girou para a **direita** (pan direita)  
- `cam_pos_y_px > 0` → câmera inclinou para **baixo** (coordenadas de imagem)  
- `bg_dx_px` é o deslocamento do fundo (sinal oposto à câmera)

## Dependências

```bash
pip install opencv-python numpy pandas matplotlib scipy
```

In [ ]:
from __future__ import annotations

import math
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection

try:
    from scipy.signal import savgol_filter
    SCIPY_OK = True
except ImportError:
    SCIPY_OK = False
    print("[aviso] scipy não encontrado. Instale com: pip install scipy")
    print("         Usando numpy.gradient como fallback (mais ruidoso).")

try:
    import cv2
except ImportError as _e:
    raise ImportError("OpenCV necessário: pip install opencv-python") from _e

print(f"OpenCV {cv2.__version__}  |  scipy disponível: {SCIPY_OK}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# FUNÇÕES AUXILIARES
# ═══════════════════════════════════════════════════════════════════

def _safe_fps(cap: cv2.VideoCapture, fallback: float = 30.0) -> float:
    """Lê o FPS do vídeo com fallback seguro."""
    fps = float(cap.get(cv2.CAP_PROP_FPS) or 0.0)
    return fps if (math.isfinite(fps) and fps > 0) else fallback


def _to_gray(frame: np.ndarray) -> np.ndarray:
    """Converte frame para escala de cinza e aplica blur suave."""
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY) if frame.ndim == 3 else frame.copy()
    return cv2.GaussianBlur(gray, (3, 3), 0)


def _highpass(gray: np.ndarray, sigma: float = 5.0) -> np.ndarray:
    """
    Filtro passa-alta para melhorar a correlação de fase em imagens térmicas.
    Remove a componente DC (brilho médio) e normaliza a variância.
    """
    gray_f = gray.astype(np.float32)
    blurred = cv2.GaussianBlur(gray_f, (0, 0), sigma)
    hp = gray_f - blurred
    std = float(np.std(hp))
    return hp / (std + 1e-6)


def _build_hud_mask(
    shape: tuple[int, int],
    ignore_reticle: bool = True,
    reticle_center_fraction: float = 0.18,
    reticle_line_fraction: float = 0.02,
    hud_boxes: list[tuple[int, int, int, int]] | None = None,
) -> np.ndarray:
    """
    Gera uma máscara binária (255 = background válido, 0 = excluído) que remove:

    - As linhas do crosshair/retículo (fixas na imagem pelo sistema de armas)
    - A caixa central ao redor do alvo (região de alto contraste)
    - Quaisquer caixas HUD adicionais informadas pelo usuário

    Parâmetros
    ----------
    shape : (height, width)
    ignore_reticle : mascara o crosshair e a caixa central se True
    reticle_center_fraction : fração das dimensões do frame a mascarar ao redor do centro
    reticle_line_fraction : espessura relativa das bandas horizontal/vertical do crosshair
    hud_boxes : lista de (x, y, w, h) em pixels de regiões fixas adicionais a excluir
    """
    h, w = shape[:2]
    mask = np.ones((h, w), dtype=np.uint8) * 255
    cx, cy = w / 2.0, h / 2.0

    if ignore_reticle:
        # Banda horizontal do crosshair
        lh = max(1, int(round(h * reticle_line_fraction / 2)))
        mask[max(0, int(cy) - lh) : min(h, int(cy) + lh + 1), :] = 0

        # Banda vertical do crosshair
        lw = max(1, int(round(w * reticle_line_fraction / 2)))
        mask[:, max(0, int(cx) - lw) : min(w, int(cx) + lw + 1)] = 0

        # Caixa central (onde o alvo está sendo rastreado)
        half_w = max(1, int(round(w * reticle_center_fraction / 2)))
        half_h = max(1, int(round(h * reticle_center_fraction / 2)))
        mask[
            max(0, int(cy) - half_h) : min(h, int(cy) + half_h + 1),
            max(0, int(cx) - half_w) : min(w, int(cx) + half_w + 1),
        ] = 0

    # Caixas HUD adicionais (texto, numeração, etc.)
    for box in (hud_boxes or []):
        x, y, bw, bh = (int(v) for v in box)
        mask[max(0, y) : min(h, y + bh), max(0, x) : min(w, x + bw)] = 0

    return mask


def _refine_mask_with_texture(
    prev_gray: np.ndarray,
    curr_gray: np.ndarray,
    base_mask: np.ndarray,
    min_intensity: int = 8,
    max_intensity: int = 247,
    texture_percentile: float = 45.0,
) -> tuple[np.ndarray, float]:
    """
    Refina a máscara HUD removendo também:

    - Pixels saturados (brilho máximo) ou negros — típicos de sobreposições FLIR
    - Regiões sem textura suficiente para estimar movimento com confiança

    Retorna (máscara_refinada, fração_de_cobertura)
    """
    mask = base_mask.copy()

    # Remove pixels saturados e negros
    intensity_ok = (
        (prev_gray >= min_intensity) & (prev_gray <= max_intensity) &
        (curr_gray >= min_intensity) & (curr_gray <= max_intensity)
    ).astype(np.uint8) * 255
    mask = cv2.bitwise_and(mask, intensity_ok)

    # Requer textura (bordas detectadas por Sobel)
    sx = cv2.Sobel(prev_gray, cv2.CV_32F, 1, 0, ksize=3)
    sy = cv2.Sobel(prev_gray, cv2.CV_32F, 0, 1, ksize=3)
    texture = cv2.magnitude(sx, sy)
    valid_pixels = texture[mask > 0]
    thresh = max(float(np.percentile(valid_pixels, texture_percentile)), 2.0) if valid_pixels.size > 100 else 2.0
    texture_ok = (texture >= thresh).astype(np.uint8) * 255
    mask = cv2.bitwise_and(mask, texture_ok)

    # Limpeza morfológica
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))

    coverage = float(np.count_nonzero(mask)) / mask.size
    return mask, coverage


def _estimate_translation_phase(
    prev_gray: np.ndarray,
    curr_gray: np.ndarray,
    mask: np.ndarray,
    min_coverage: float = 0.04,
    min_response: float = 0.012,
) -> tuple[float, float, float, bool]:
    """
    Estima a translação entre frames usando correlação de fase no domínio da frequência.

    A imagem é pré-filtrada com um filtro passa-alta e a máscara de background é aplicada
    antes da correlação, eliminando o efeito de regiões planas (céu uniforme, HUD).

    Retorna (dx_px, dy_px, phase_response, sucesso)
    """
    coverage = float(np.count_nonzero(mask)) / mask.size
    if coverage < min_coverage:
        return 0.0, 0.0, 0.0, False

    mask_f = (mask > 0).astype(np.float32)
    hp_prev = _highpass(prev_gray) * mask_f
    hp_curr = _highpass(curr_gray) * mask_f

    hanning = cv2.createHanningWindow((prev_gray.shape[1], prev_gray.shape[0]), cv2.CV_32F)
    try:
        (dx, dy), response = cv2.phaseCorrelate(hp_prev, hp_curr, hanning)
    except cv2.error:
        return 0.0, 0.0, 0.0, False

    if not (math.isfinite(dx) and math.isfinite(dy) and math.isfinite(response)):
        return 0.0, 0.0, 0.0, False

    ok = float(response) >= min_response
    return float(dx), float(dy), float(response), ok


def _estimate_translation_dense(
    prev_gray: np.ndarray,
    curr_gray: np.ndarray,
    mask: np.ndarray,
    min_valid_pixels: int = 200,
) -> tuple[float, float, bool]:
    """
    Estima a translação via fluxo óptico denso de Farnebäck.
    Usado como fallback quando a correlação de fase falha.

    Retorna (dx_px, dy_px, sucesso)
    """
    flow = cv2.calcOpticalFlowFarneback(
        prev_gray, curr_gray, None,
        pyr_scale=0.5, levels=4, winsize=25,
        iterations=3, poly_n=5, poly_sigma=1.2, flags=0,
    )
    mag = cv2.magnitude(flow[..., 0], flow[..., 1])
    max_expected = max(prev_gray.shape[:2]) * 0.3
    valid = (mask > 0) & np.isfinite(flow[..., 0]) & (mag < max_expected)
    if int(np.count_nonzero(valid)) < min_valid_pixels:
        return 0.0, 0.0, False
    return float(np.median(flow[..., 0][valid])), float(np.median(flow[..., 1][valid])), True


def _make_sg_window(n_frames: int, smooth_window_s: float, eff_fps: float, poly: int) -> int:
    """Calcula um tamanho de janela válido para o filtro Savitzky-Golay (deve ser ímpar)."""
    window = max(poly + 2, int(round(smooth_window_s * eff_fps)))
    if window % 2 == 0:
        window += 1
    max_window = n_frames if n_frames % 2 != 0 else n_frames - 1
    window = min(window, max(poly + 1, max_window))
    return max(window, poly + 1 if (poly + 1) % 2 != 0 else poly + 2)


def _smooth_and_derive(
    values: np.ndarray,
    dt_s: float,
    window: int,
    poly: int,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Aplica o filtro Savitzky-Golay para obter:
    - Posição suavizada
    - Velocidade (1ª derivada analítica do filtro)
    - Aceleração (2ª derivada analítica do filtro)

    O SG calcula derivadas analiticamente no polinômio ajustado, o que produz
    estimativas mais suaves do que diferenciar numericamente após suavização.

    Retorna (posição_suave, velocidade_px_s, aceleração_px_s2)
    """
    n = len(values)
    if not SCIPY_OK or n < window:
        # Fallback: média móvel + gradiente numérico
        kernel = np.ones(min(window, n)) / min(window, n)
        smooth = np.convolve(values, kernel, mode='same')
        vel = np.gradient(smooth, dt_s)
        acc = np.gradient(vel, dt_s)
        return smooth, vel, acc

    smooth = savgol_filter(values, window, poly)
    vel    = savgol_filter(values, window, poly, deriv=1, delta=dt_s)
    acc    = savgol_filter(values, window, poly, deriv=2, delta=dt_s)
    return smooth, vel, acc


print("Funções auxiliares carregadas.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# FUNÇÃO PRINCIPAL
# ═══════════════════════════════════════════════════════════════════

def analyze_camera_motion(
    video_path: str | Path,
    *,
    # ── Máscara de HUD ─────────────────────────────────────────────
    ignore_reticle: bool = True,
    reticle_center_fraction: float = 0.18,
    reticle_line_fraction: float = 0.02,
    hud_boxes: list[tuple[int, int, int, int]] | None = None,
    # ── Estimativa de movimento ────────────────────────────────────
    method: str = "auto",
    frame_step: int = 1,
    roi: tuple[int, int, int, int] | None = None,
    # ── Suavização (Savitzky-Golay) ────────────────────────────────
    smooth_window_s: float = 0.4,
    smooth_poly: int = 3,
    # ── Misc ───────────────────────────────────────────────────────
    fps_override: float | None = None,
    verbose: bool = True,
) -> dict[str, Any]:
    """
    Estima posição, velocidade e aceleração da câmera FLIR a partir do
    movimento do **fundo** do vídeo, mascarando elementos fixos da HUD.

    Parâmetros
    ----------
    video_path : str | Path
        Caminho para o arquivo de vídeo (qualquer formato suportado pelo OpenCV).
    ignore_reticle : bool, padrão True
        Mascara as linhas do crosshair e a caixa central antes de estimar movimento.
        Deve ser True para vídeos de targeting pod / câmera de armas.
    reticle_center_fraction : float, padrão 0.18
        Fração das dimensões do frame a mascarar ao redor do centro (onde está o alvo).
        Aumente se o alvo ocupar uma área maior do centro.
    reticle_line_fraction : float, padrão 0.02
        Espessura relativa das bandas horizontal e vertical do crosshair.
    hud_boxes : list of (x, y, w, h), opcional
        Regiões fixas adicionais a excluir da estimativa de movimento
        (e.g. altitude, velocidade airspeed, designadores de texto nos cantos da HUD).
        Coordenadas em pixels do frame completo (ou do roi, se fornecido).
    method : {'auto', 'phase', 'dense'}, padrão 'auto'
        Método de estimativa de movimento por frame:
        - 'phase' : correlação de fase com filtro passa-alta (rápido, preciso).
        - 'dense' : fluxo óptico denso de Farnebäck (mais robusto em texturas pobres).
        - 'auto'  : tenta phase primeiro; usa dense se a resposta for baixa.
    frame_step : int, padrão 1
        Processa 1 a cada N frames (1 = todos os frames).
    roi : (x, y, w, h), opcional
        Recorta o frame para esta região antes de analisar.
        Útil para excluir bordas com artefatos.
    smooth_window_s : float, padrão 0.4
        Largura da janela Savitzky-Golay em segundos.
        Valores maiores → curvas mais suaves, mas derivadas menos resolvidas.
    smooth_poly : int, padrão 3
        Ordem do polinômio Savitzky-Golay. Deve ser < janela em frames.
    fps_override : float, opcional
        Sobrescreve o FPS reportado pelo vídeo (útil para vídeos com metadados errados).
    verbose : bool, padrão True
        Imprime informações de progresso.

    Retorna
    -------
    dict com chaves:
        'metadata' : dict com parâmetros do vídeo e da análise.
        'data'     : pd.DataFrame com dados de movimento por frame.

    Colunas principais do DataFrame
    --------------------------------
    frame_index, time_s
        Índice e tempo do frame processado.
    bg_dx_px, bg_dy_px
        Deslocamento do **fundo** por frame (sinal oposto à câmera).
        bg_dx_px > 0 → fundo se deslocou para a direita → câmera girou para a esquerda.
    cam_pos_x_px, cam_pos_y_px
        Posição **cumulativa** da câmera (cam_pos = -bg cumulativo).
        cam_pos_x > 0 → câmera girou para a direita (pan direita).
        cam_pos_y > 0 → câmera inclinou para baixo (em coordenadas de imagem).
    cam_pos_x_smooth_px, cam_pos_y_smooth_px
        Posição suavizada pelo filtro Savitzky-Golay.
    cam_vel_x_px_s, cam_vel_y_px_s
        Velocidade da câmera em px/s (1ª derivada analítica do SG).
    cam_acc_x_px_s2, cam_acc_y_px_s2
        Aceleração da câmera em px/s² (2ª derivada analítica do SG).
    cam_speed_px_s
        Módulo da velocidade (norma de vx e vy).
    cam_accel_magnitude_px_s2
        Módulo da aceleração.
    phase_response
        Qualidade da correlação de fase (> 0.012 = confiável).
    mask_coverage
        Fração do frame usada como background (0–1).
    motion_method
        Método utilizado ('phase', 'dense', 'none', 'init').
    """
    video_path = Path(video_path).expanduser().resolve()
    if not video_path.exists():
        raise FileNotFoundError(f"Vídeo não encontrado: {video_path}")
    if frame_step < 1:
        raise ValueError("frame_step deve ser >= 1")
    method = method.lower()
    if method not in {"auto", "phase", "dense"}:
        raise ValueError("method deve ser 'auto', 'phase' ou 'dense'")

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise ValueError(f"Não foi possível abrir o vídeo: {video_path}")

    fps = fps_override if fps_override else _safe_fps(cap)
    n_nominal = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    vid_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH) or 0)
    vid_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT) or 0)

    if verbose:
        print(f"Vídeo : {video_path.name}")
        print(f"       {vid_w}×{vid_h} px  |  {fps:.3f} fps  |  ~{n_nominal} frames")

    # Validar ROI
    if roi is not None:
        rx, ry, rw, rh = (int(v) for v in roi)
        if rx < 0 or ry < 0 or rw <= 0 or rh <= 0 or rx + rw > vid_w or ry + rh > vid_h:
            raise ValueError(f"roi {roi} inválido para frame {vid_w}×{vid_h}")
    else:
        rx, ry, rw, rh = 0, 0, vid_w, vid_h

    dt_s = frame_step / fps          # intervalo de tempo entre frames processados
    eff_fps = fps / frame_step       # FPS efetivo após frame_step

    rows: list[dict[str, Any]] = []
    prev_gray: np.ndarray | None = None
    read_idx = 0

    while True:
        ok, frame = cap.read()
        if not ok:
            break

        fi = read_idx
        read_idx += 1

        if fi % frame_step != 0:
            continue

        time_s = fi / fps
        cropped = frame[ry : ry + rh, rx : rx + rw]
        gray = _to_gray(cropped)

        # Primeiro frame: inicializa apenas
        if prev_gray is None:
            prev_gray = gray
            rows.append({
                "frame_index": fi, "time_s": time_s,
                "bg_dx_px": 0.0, "bg_dy_px": 0.0,
                "phase_response": np.nan, "mask_coverage": np.nan,
                "motion_method": "init", "motion_ok": False,
            })
            continue

        # ── Construir máscara de HUD ───────────────────────────────
        hud_mask = _build_hud_mask(
            gray.shape,
            ignore_reticle=ignore_reticle,
            reticle_center_fraction=reticle_center_fraction,
            reticle_line_fraction=reticle_line_fraction,
            hud_boxes=hud_boxes,
        )

        # ── Refinar máscara com análise de textura ─────────────────
        bg_mask, coverage = _refine_mask_with_texture(prev_gray, gray, hud_mask)

        # ── Estimar translação ─────────────────────────────────────
        bg_dx, bg_dy = 0.0, 0.0
        phase_response = np.nan
        motion_method = "none"
        motion_ok = False

        if method in {"auto", "phase"}:
            bg_dx, bg_dy, phase_response, ok_p = _estimate_translation_phase(
                prev_gray, gray, bg_mask
            )
            if ok_p:
                motion_method = "phase"
                motion_ok = True

        if not motion_ok and method in {"auto", "dense"}:
            bg_dx, bg_dy, ok_d = _estimate_translation_dense(prev_gray, gray, bg_mask)
            if ok_d:
                motion_method = "dense"
                motion_ok = True

        rows.append({
            "frame_index": fi, "time_s": time_s,
            "bg_dx_px": bg_dx, "bg_dy_px": bg_dy,
            "phase_response": phase_response,
            "mask_coverage": coverage,
            "motion_method": motion_method,
            "motion_ok": motion_ok,
        })
        prev_gray = gray

    cap.release()

    if not rows:
        raise RuntimeError("Nenhum frame foi processado.")

    df = pd.DataFrame(rows)
    n = len(df)

    # ── Posição cumulativa ─────────────────────────────────────────────────
    # bg_dx > 0: fundo se deslocou para a direita → câmera girou para a esquerda
    # cam_pos = -cumsum(bg_dx) → positivo quando câmera girou para a direita
    df["cam_pos_x_px"] = -df["bg_dx_px"].fillna(0).cumsum()
    df["cam_pos_y_px"] = -df["bg_dy_px"].fillna(0).cumsum()

    # Também mantém o deslocamento do fundo (convenção oposta)
    df["bg_pos_x_px"] = df["bg_dx_px"].fillna(0).cumsum()
    df["bg_pos_y_px"] = df["bg_dy_px"].fillna(0).cumsum()

    # ── Savitzky-Golay: posição suave + velocidade + aceleração ───────────
    sg_window = _make_sg_window(n, smooth_window_s, eff_fps, smooth_poly)

    if verbose:
        print(f"\nFrames processados  : {n}")
        print(f"Janela SG           : {sg_window} frames ({sg_window * dt_s:.2f} s)")
        ok_count = int(df["motion_ok"].sum())
        ok_pct = 100 * ok_count / max(n - 1, 1)
        print(f"Movimento estimado  : {ok_count}/{n-1} intervals ({ok_pct:.1f}%)")
        print("Método (contagem)   :", df.groupby("motion_method").size().to_dict())

    pos_x = df["cam_pos_x_px"].to_numpy(dtype=float)
    pos_y = df["cam_pos_y_px"].to_numpy(dtype=float)

    smooth_x, vel_x, acc_x = _smooth_and_derive(pos_x, dt_s, sg_window, smooth_poly)
    smooth_y, vel_y, acc_y = _smooth_and_derive(pos_y, dt_s, sg_window, smooth_poly)

    df["cam_pos_x_smooth_px"]    = smooth_x
    df["cam_pos_y_smooth_px"]    = smooth_y
    df["cam_vel_x_px_s"]         = vel_x
    df["cam_vel_y_px_s"]         = vel_y
    df["cam_acc_x_px_s2"]        = acc_x
    df["cam_acc_y_px_s2"]        = acc_y
    df["cam_speed_px_s"]         = np.hypot(vel_x, vel_y)
    df["cam_accel_magnitude_px_s2"] = np.hypot(acc_x, acc_y)

    metadata = {
        "video_path": str(video_path),
        "fps": fps, "frame_step": frame_step, "effective_fps": eff_fps, "dt_s": dt_s,
        "video_width": vid_w, "video_height": vid_h,
        "roi": (rx, ry, rw, rh),
        "ignore_reticle": ignore_reticle,
        "reticle_center_fraction": reticle_center_fraction,
        "reticle_line_fraction": reticle_line_fraction,
        "hud_boxes": hud_boxes or [],
        "method": method,
        "smooth_window_s": smooth_window_s,
        "smooth_poly": smooth_poly,
        "sg_window_frames": sg_window,
        "n_frames_processed": n,
        "n_frames_motion_ok": int(df["motion_ok"].sum()),
    }

    return {"metadata": metadata, "data": df}


print("analyze_camera_motion() carregada.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# FUNÇÕES DE PLOTAGEM
# ═══════════════════════════════════════════════════════════════════

def plot_camera_kinematics(
    result: dict[str, Any],
    figsize: tuple[float, float] = (15, 11),
    title: str | None = None,
    show_raw: bool = True,
) -> plt.Figure:
    """
    Plota posição, velocidade e aceleração da câmera em X e Y.

    Layout: 3 linhas × 2 colunas
      Linha 1 — Posição X(t) e Y(t) [px]  (raw + suavizado)
      Linha 2 — Velocidade X(t) e Y(t) [px/s]
      Linha 3 — Aceleração X(t) e Y(t) [px/s²]

    Parâmetros
    ----------
    result   : saída de analyze_camera_motion()
    figsize  : tamanho da figura matplotlib
    title    : título opcional (padrão: nome do arquivo de vídeo)
    show_raw : exibe a curva raw (não suavizada) de posição junto com a suavizada

    Retorna
    -------
    matplotlib.figure.Figure
    """
    df   = result["data"]
    meta = result["metadata"]
    t    = df["time_s"].to_numpy()

    fig, axes = plt.subplots(3, 2, figsize=figsize, sharex=True)
    fig.suptitle(
        f"Cinemática da Câmera FLIR — {title or Path(meta['video_path']).name}",
        fontsize=13, fontweight="bold", y=1.01,
    )

    # Paleta de cores por eixo
    cx, cy = "#1f77b4", "#17becf"   # azul / ciano para X e Y de posição
    vx, vy = "#ff7f0e", "#d62728"   # laranja / vermelho para velocidade
    ax_c, ay_c = "#2ca02c", "#9467bd"  # verde / roxo para aceleração

    # ── Linha 1: Posição ───────────────────────────────────────────
    for col, (raw_col, smooth_col, color, label, subtitle) in enumerate([
        ("cam_pos_x_px", "cam_pos_x_smooth_px", cx, "X", "Camera Pan — X  (+ = direita)"),
        ("cam_pos_y_px", "cam_pos_y_smooth_px", cy, "Y", "Camera Tilt — Y  (+ = baixo)"),
    ]):
        ax = axes[0, col]
        if show_raw:
            ax.plot(t, df[raw_col], color=color, alpha=0.25, lw=1, label="raw")
        ax.plot(t, df[smooth_col], color=color, lw=2.2, label="suavizado (SG)")
        ax.axhline(0, color="black", lw=0.8, alpha=0.35)
        ax.set_ylabel(f"Posição {label} (px)", fontsize=10)
        ax.set_title(subtitle, fontsize=9)
        ax.legend(fontsize=8, loc="upper right")
        ax.grid(True, alpha=0.3)

    # ── Linha 2: Velocidade ────────────────────────────────────────
    for col, (vel_col, color, label) in enumerate([
        ("cam_vel_x_px_s", vx, "X"),
        ("cam_vel_y_px_s", vy, "Y"),
    ]):
        ax = axes[1, col]
        ax.plot(t, df[vel_col], color=color, lw=1.8)
        ax.axhline(0, color="black", lw=0.8, alpha=0.35)
        ax.set_ylabel(f"Velocidade {label} (px/s)", fontsize=10)
        ax.set_title(f"Velocidade da Câmera — {label}", fontsize=9)
        ax.grid(True, alpha=0.3)

    # ── Linha 3: Aceleração ────────────────────────────────────────
    for col, (acc_col, color, label) in enumerate([
        ("cam_acc_x_px_s2", ax_c, "X"),
        ("cam_acc_y_px_s2", ay_c, "Y"),
    ]):
        ax = axes[2, col]
        ax.plot(t, df[acc_col], color=color, lw=1.8)
        ax.axhline(0, color="black", lw=0.8, alpha=0.35)
        ax.set_ylabel(f"Aceleração {label} (px/s²)", fontsize=10)
        ax.set_xlabel("Tempo (s)", fontsize=10)
        ax.set_title(f"Aceleração da Câmera — {label}", fontsize=9)
        ax.grid(True, alpha=0.3)

    # Anotar parâmetros relevantes no rodapé
    footer = (
        f"FPS efetivo: {meta['effective_fps']:.1f}  |  "
        f"Janela SG: {meta['sg_window_frames']} frames ({meta['smooth_window_s']:.2f} s)  |  "
        f"Método: {meta['method']}  |  "
        f"OK: {meta['n_frames_motion_ok']}/{meta['n_frames_processed']-1} inter-frames"
    )
    fig.text(0.5, -0.01, footer, ha="center", fontsize=8, color="gray")

    fig.tight_layout()
    return fig


def plot_camera_trajectory(
    result: dict[str, Any],
    figsize: tuple[float, float] = (7, 6),
) -> plt.Figure:
    """
    Plota a trajetória 2D da câmera no plano da imagem.
    A trajetória é colorida por tempo (início = roxo escuro, fim = amarelo).

    Retorna
    -------
    matplotlib.figure.Figure
    """
    df   = result["data"]
    meta = result["metadata"]
    x    = df["cam_pos_x_smooth_px"].to_numpy()
    y    = df["cam_pos_y_smooth_px"].to_numpy()
    t    = df["time_s"].to_numpy()

    fig, ax = plt.subplots(figsize=figsize)

    # Trajetória colorida por tempo via LineCollection
    points   = np.stack([x, y], axis=1).reshape(-1, 1, 2)
    segments = np.concatenate([points[:-1], points[1:]], axis=1)
    lc = LineCollection(segments, cmap="plasma", linewidth=2)
    lc.set_array(t[:-1])
    ax.add_collection(lc)
    cbar = fig.colorbar(lc, ax=ax)
    cbar.set_label("Tempo (s)", fontsize=10)

    ax.scatter(x[0],  y[0],  color="royalblue", s=90, zorder=5, label="Início", marker="o")
    ax.scatter(x[-1], y[-1], color="crimson",   s=90, zorder=5, label="Fim",    marker="*")

    ax.set_xlabel("Deslocamento câmera X (px)  [+ = pan direita]", fontsize=10)
    ax.set_ylabel("Deslocamento câmera Y (px)  [+ = tilt baixo]",  fontsize=10)
    ax.set_title(
        f"Trajetória 2D da Câmera\n{Path(meta['video_path']).name}",
        fontsize=11,
    )
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.autoscale()
    ax.set_aspect("equal", adjustable="datalim")
    fig.tight_layout()
    return fig


def plot_motion_quality(
    result: dict[str, Any],
    figsize: tuple[float, float] = (14, 5),
) -> plt.Figure:
    """
    Plota métricas de qualidade da estimativa de movimento:
    - Resposta da correlação de fase por frame
    - Cobertura do background (fração do frame usada)
    - Velocidade total da câmera (px/s)

    Quadros com baixa qualidade podem indicar regiões onde a câmera
    aponta para céu uniforme ou regiões mascaradas demais.

    Retorna
    -------
    matplotlib.figure.Figure
    """
    df = result["data"]
    t  = df["time_s"].to_numpy()

    fig, axes = plt.subplots(1, 3, figsize=figsize)
    fig.suptitle("Qualidade da Estimativa de Movimento da Câmera", fontsize=12, fontweight="bold")

    ax = axes[0]
    ax.plot(t, df["phase_response"], color="#1f77b4", lw=1.5, label="resposta")
    ax.axhline(0.012, color="crimson", ls="--", lw=1.2, label="limiar mínimo (0.012)")
    ax.set_xlabel("Tempo (s)"); ax.set_ylabel("Resposta de correlação de fase")
    ax.set_title("Phase Response"); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

    ax = axes[1]
    ax.plot(t, df["mask_coverage"] * 100, color="#2ca02c", lw=1.5)
    ax.axhline(4, color="crimson", ls="--", lw=1.2, label="mínimo (4%)")
    ax.set_xlabel("Tempo (s)"); ax.set_ylabel("Cobertura background (%)")
    ax.set_title("Cobertura da Máscara"); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

    ax = axes[2]
    ax.plot(t, df["cam_speed_px_s"], color="#ff7f0e", lw=1.5)
    ax.set_xlabel("Tempo (s)"); ax.set_ylabel("Velocidade total câmera (px/s)")
    ax.set_title("Velocidade da Câmera"); ax.grid(True, alpha=0.3)

    fig.tight_layout()
    return fig


def plot_all(result: dict[str, Any]) -> None:
    """Gera e exibe os três gráficos de uma vez."""
    plot_camera_kinematics(result)
    plt.show()
    plot_camera_trajectory(result)
    plt.show()
    plot_motion_quality(result)
    plt.show()


print("Funções de plotagem carregadas.")

## Como Usar

### Chamada mínima
```python
result = analyze_camera_motion("meu_video_flir.mp4")
plot_all(result)
```

### Parâmetros mais usados

| Parâmetro | Quando ajustar |
|---|---|
| `reticle_center_fraction` | Aumente (ex: `0.25`) se o alvo/mira ocupa mais área central |
| `hud_boxes` | Liste caixas `(x, y, w, h)` de texto HUD fixo nos cantos |
| `smooth_window_s` | Aumente para suavizar mais; diminua para preservar detalhes rápidos |
| `roi` | Recorte a região de interesse para excluir bordas com artefatos |
| `method='dense'` | Use se a câmera faz pans muito rápidos ou cena com pouca textura |
| `frame_step=2` | Use para vídeos longos ou quando o processamento está lento |

### Interpretação dos gráficos

- **Posição**: deslocamento cumulativo da câmera. Pan contínuo aparece como rampa.
- **Velocidade**: taxa de pan/tilt. Picos indicam manobras bruscas.
- **Aceleração**: mudança da velocidade. Picos altos indicam manobras abruptas do sistema de armas.
- **Phase Response < 0.012**: frame problemático — a estimativa de movimento pode ser imprecisa.
- **Cobertura < 4%**: background insuficiente — aumente `reticle_center_fraction` ou ajuste o `roi`.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# EXEMPLO DE USO
# Ajuste o caminho do vídeo e os parâmetros conforme necessário.
# ═══════════════════════════════════════════════════════════════════

VIDEO_PATH = "/content/seu_video_flir.mp4"   # ← altere para o caminho do vídeo

result = analyze_camera_motion(
    VIDEO_PATH,

    # ── Máscara de HUD ────────────────────────────────────────────
    ignore_reticle=True,
    reticle_center_fraction=0.18,   # fração do frame ao redor do centro a mascarar
    reticle_line_fraction=0.02,     # espessura das linhas do crosshair
    hud_boxes=None,                 # ex: [(0, 0, 120, 40), (520, 0, 120, 40)]
                                    #       ^ canto sup-esq    ^ canto sup-dir

    # ── Estimativa de movimento ───────────────────────────────────
    method="auto",                  # 'auto' | 'phase' | 'dense'
    frame_step=1,                   # 1 = todos os frames
    roi=None,                       # ex: (50, 50, 600, 400) para recortar bordas

    # ── Suavização ────────────────────────────────────────────────
    smooth_window_s=0.4,            # janela SG em segundos
    smooth_poly=3,                  # grau do polinômio SG

    verbose=True,
)

# ── Resumo numérico ───────────────────────────────────────────────
df = result["data"]
print("\n── Estatísticas da câmera ──")
print(df[["cam_vel_x_px_s", "cam_vel_y_px_s",
          "cam_acc_x_px_s2", "cam_acc_y_px_s2",
          "cam_speed_px_s"]].describe().round(2))

In [ ]:
# ── Gráfico 1: Posição, Velocidade e Aceleração ───────────────────
fig1 = plot_camera_kinematics(result, show_raw=True)
plt.show()

In [ ]:
# ── Gráfico 2: Trajetória 2D da câmera ───────────────────────────
fig2 = plot_camera_trajectory(result)
plt.show()

In [ ]:
# ── Gráfico 3: Métricas de qualidade da estimativa ────────────────
fig3 = plot_motion_quality(result)
plt.show()

In [ ]:
# ── Exportar DataFrame para CSV (opcional) ────────────────────────
# result["data"].to_csv("camera_motion.csv", index=False)
# print("Dados exportados para camera_motion.csv")

# ── Inspecionar colunas disponíveis ──────────────────────────────
print("Colunas disponíveis no DataFrame:")
for col in result["data"].columns:
    print(f"  {col}")